In [1]:
# 升级 transformers 库以支持 vaultgemma 架构
# 使用 pip install -q 表示“静默安装”，减少输出，保持 Notebook 界面整洁
!pip install -q --upgrade transformers

# 检查升级后的版本（可选）
import transformers
print(f"Transformers version after upgrade: {transformers.__version__}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.1.1 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.0a1 which is incompatible.
Transformers version after upgrade: 4.57.1


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


/kaggle/input/digit-recognizer/sample_submission.csv
/kaggle/input/digit-recognizer/train.csv
/kaggle/input/digit-recognizer/test.csv
/kaggle/input/gemma/transformers/1.1-2b-it/1/model.safetensors.index.json
/kaggle/input/gemma/transformers/1.1-2b-it/1/config.json
/kaggle/input/gemma/transformers/1.1-2b-it/1/model-00001-of-00002.safetensors
/kaggle/input/gemma/transformers/1.1-2b-it/1/model-00002-of-00002.safetensors
/kaggle/input/gemma/transformers/1.1-2b-it/1/tokenizer.json
/kaggle/input/gemma/transformers/1.1-2b-it/1/tokenizer_config.json
/kaggle/input/gemma/transformers/1.1-2b-it/1/special_tokens_map.json
/kaggle/input/gemma/transformers/1.1-2b-it/1/.gitattributes
/kaggle/input/gemma/transformers/1.1-2b-it/1/tokenizer.model
/kaggle/input/gemma/transformers/1.1-2b-it/1/generation_config.json


In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# --- 环境设置提醒 (在 Notebook 中，请确保这些命令已在单独单元格中运行) ---
# !pip install -q --upgrade transformers
# !pip install -q git+https://github.com/huggingface/transformers.git

# --- 1. 定义模型和数据文件的路径 ---
# 模型路径 (Gemma 1B) - 尽管加载失败，仍保留路径定义
MODEL_PATH = "/kaggle/input/vaultgemma/transformers/1b/1"

# 数字识别数据 - 来自 'digit-recognizer-challenge' (主要使用这个数据集)
CHALLENGE_TRAIN_PATH = "/kaggle/input/digit-recognizer/train.csv"
CHALLENGE_TEST_PATH = "/kaggle/input/digit-recognizer/test.csv"

print("✅ 所有输入路径已定义。")
print("-" * 40)


# --- 2. 加载数据 ---
print("📊 正在加载数字识别数据...")
try:
    df_train_challenge = pd.read_csv(CHALLENGE_TRAIN_PATH)
    df_test_challenge = pd.read_csv(CHALLENGE_TEST_PATH)
    print(f"  - 训练集形状: {df_train_challenge.shape}")
    print(f"  - 测试集形状: {df_test_challenge.shape}")
except FileNotFoundError as e:
    print(f"❌ 错误：无法找到数据文件。请检查路径。错误信息: {e}")
    exit() # 如果数据加载失败，终止脚本

print("-" * 40)


# --- 3. 尝试加载 Gemma 模型 (当前已确认失败，但保留代码块) ---
print("🧠 正在尝试加载 Gemma 模型...")
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    # 强制将模型路径添加到系统路径，并尝试使用 trust_remote_code
    import sys
    sys.path.append(MODEL_PATH) 
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    model_gemma = AutoModelForCausalLM.from_pretrained(MODEL_PATH, trust_remote_code=True)
    print("  - Gemma 模型和分词器加载成功。")

except Exception as e:
    # 忽略此警告，继续执行数字识别任务
    print(f"⚠️ 警告：Gemma 模型加载失败 (预期错误)。错误信息: {e.args[0][:120]}...")

print("-" * 40)


# --- 4. 数字识别数据预处理 ---
print("⚙️ 正在进行数据预处理...")

# 4.1 分离特征 X 和标签 y
X_train = df_train_challenge.drop('label', axis=1).values # 特征 (像素数据)
y_train = df_train_challenge['label'].values             # 标签 (数字 0-9)
X_test = df_test_challenge.values                        # 测试集特征

# 4.2 数据标准化/归一化 (Normalization)
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# 4.3 重塑数据为图像矩阵 (为 CNN 模型准备)
# 形状: (样本数, 高度, 宽度, 通道数)
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# 4.4 对标签进行独热编码 (One-Hot Encoding)
y_train_encoded = to_categorical(y_train, num_classes=10)

print("✅ 数据预处理完成，数据形状已调整。")
print(f"X_train 最终形状: {X_train.shape}, y_train_encoded 最终形状: {y_train_encoded.shape}")

print("-" * 40)


# --- 5. 构建并编译 CNN 模型 ---
print("🏗️ 正在构建和编译 CNN 模型...")
cnn_model = Sequential([
    # 卷积层 1
    Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D(pool_size=(2, 2)),
    
    # 卷积层 2
    Conv2D(64, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    
    # 展平层
    Flatten(),
    
    # 全连接层
    Dense(128, activation='relu'),
    Dropout(0.5), 
    
    # 输出层
    Dense(10, activation='softmax')
])

cnn_model.compile(optimizer='adam', 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])

print("✅ CNN 模型架构已定义并编译。")
cnn_model.summary()

print("-" * 40)


# --- 6. 模型训练 ---
print("🏃 正在开始模型训练 (10 Epochs)...")
history = cnn_model.fit(
    X_train, y_train_encoded,
    batch_size=128,      
    epochs=10,           
    validation_split=0.1 
)
print("\n🎉 模型训练成功结束。")

print("-" * 40)


# --- 7. 生成 Kaggle 提交文件 ---
print("📝 正在生成 Kaggle 提交文件...")

# 7.1 使用模型进行预测
# model.predict() 会返回一个概率矩阵
predictions_prob = cnn_model.predict(X_test)

# 7.2 获取最终的预测标签 (数字)
# np.argmax(axis=1) 获取概率最高的索引
predictions = np.argmax(predictions_prob, axis=1)

# 7.3 创建 ImageId 序列 (从 1 开始)
image_ids = np.arange(1, len(predictions) + 1)

# 7.4 创建 DataFrame
submission_df = pd.DataFrame({
    'ImageId': image_ids,
    'Label': predictions
})

# 7.5 导出 CSV 文件
SUBMISSION_FILE_PATH = 'submission.csv'
submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)

print(f"\n✅ 提交文件生成成功！文件名称: {SUBMISSION_FILE_PATH}")
print("\n--- 提交文件预览 (前 5 行) ---")
print(submission_df.head())
print("-" * 40)
print("✨ 整个程式流程已执行完毕。")


2025-10-28 16:06:44.331612: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761667604.608673      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761667604.695993      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


✅ 所有输入路径已定义。
----------------------------------------
📊 正在加载数字识别数据...
  - 训练集形状: (42000, 785)
  - 测试集形状: (28000, 784)
----------------------------------------
🧠 正在尝试加载 Gemma 模型...
⚠️ 警告：Gemma 模型加载失败 (预期错误)。错误信息: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/kaggle/input/vaultgemma/transformers/1b/1'. Use `rep...
----------------------------------------
⚙️ 正在进行数据预处理...
✅ 数据预处理完成，数据形状已调整。
X_train 最终形状: (42000, 28, 28, 1), y_train_encoded 最终形状: (42000, 10)
----------------------------------------
🏗️ 正在构建和编译 CNN 模型...
✅ CNN 模型架构已定义并编译。


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-10-28 16:07:17.532277: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

----------------------------------------
🏃 正在开始模型训练 (10 Epochs)...
Epoch 1/10
296/296 ━━━━━━━━━━━━━━━━━━━━ 21s 60ms/step - accuracy: 0.7445 - loss: 0.7922 - val_accuracy: 0.9736 - val_loss: 0.0868
Epoch 2/10
296/296 ━━━━━━━━━━━━━━━━━━━━ 17s 56ms/step - accuracy: 0.9618 - loss: 0.1275 - val_accuracy: 0.9810 - val_loss: 0.0587
Epoch 3/10
296/296 ━━━━━━━━━━━━━━━━━━━━ 17s 58ms/step - accuracy: 0.9755 - loss: 0.0834 - val_accuracy: 0.9840 - val_loss: 0.0474
Epoch 4/10
296/296 ━━━━━━━━━━━━━━━━━━━━ 16s 55ms/step - accuracy: 0.9788 - loss: 0.0695 - val_accuracy: 0.9874 - val_loss: 0.0400
Epoch 5/10
296/296 ━━━━━━━━━━━━━━━━━━━━ 16s 55ms/step - accuracy: 0.9831 - loss: 0.0566 - val_accuracy: 0.9883 - val_loss: 0.0391
Epoch 6/10
296/296 ━━━━━━━━━━━━━━━━━━━━ 16s 55ms/step - accuracy: 0.9858 - loss: 0.0488 - val_accuracy: 0.9902 - val_loss: 0.0374
Epoch 7/10
296/296 ━━━━━━━━━━━━━━━━━━━━ 16s 54ms/step - accuracy: 0.9865 - loss: 0.0419 - val_accuracy: 0.9905 - val_loss: 0.0322
Epoch 8/10
296/296 ━━━━